In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from delta import configure_spark_with_delta_pip
import os
import pandas as pd

In [0]:
builder = (
    SparkSession.builder.appName("Olist - Bronze Layer")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"),
)

In [0]:
spark = (
    SparkSession.builder
    .appName("Olist - Bronze Layer")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

In [0]:
RAW_PATH   = "/Workspace/Projeto_Framework-Rethink/data/raw"
DELTA_BASE = "/Workspace/Projeto_Framework-Rethink/delta/bronze"

In [0]:
tables = {
    "orders":      ("olist_orders_dataset.csv",         f"{DELTA_BASE}/orders"),
    "order_items": ("olist_order_items_dataset.csv",    f"{DELTA_BASE}/order_items"),
    "customers":   ("olist_customers_dataset.csv",      f"{DELTA_BASE}/customers"),
    "products":    ("olist_products_dataset.csv",       f"{DELTA_BASE}/products"),
    "sellers":     ("olist_sellers_dataset.csv",        f"{DELTA_BASE}/sellers"),
    "payments":    ("olist_order_payments_dataset.csv", f"{DELTA_BASE}/payments"),
    "reviews":     ("olist_order_reviews_dataset.csv",  f"{DELTA_BASE}/reviews"),
}

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

for name, (csv_file, _) in tables.items():
    print(f"[BRONZE] Loading: {name}...")
    pandas_df = pd.read_csv(f"{RAW_PATH}/{csv_file}")
    df = spark.createDataFrame(pandas_df)
    df = df.withColumn("ingest_timestamp", current_timestamp())
    df.write.mode("overwrite").format("delta").saveAsTable(f"bronze.{name}")
    print(f"[BRONZE] Loaded: {name}.")

[BRONZE] Loading: orders...
[BRONZE] Loaded: orders.
[BRONZE] Loading: order_items...
[BRONZE] Loaded: order_items.
[BRONZE] Loading: customers...
[BRONZE] Loaded: customers.
[BRONZE] Loading: products...
[BRONZE] Loaded: products.
[BRONZE] Loading: sellers...
[BRONZE] Loaded: sellers.
[BRONZE] Loading: payments...
[BRONZE] Loaded: payments.
[BRONZE] Loading: reviews...
[BRONZE] Loaded: reviews.


In [0]:
spark.stop()